In [8]:
from dotenv import load_dotenv
load_dotenv()

True

In [14]:
import openai, json

client = openai.OpenAI()

messages = [{
    "role": "system",
    "content": (
        "너는 영화 정보 에이전트다. 사용자의 영화 질문에 답하기 위해 제공된 도구를 적극 사용한다. "
        "영화를 언급할 때는 ID를 함께 파악해두고, 사용자가 이전 대화의 영화('그 영화', '듄' 등)를 가리키면 해당 ID로 도구를 호출한다."
        "영화 목록이나 추천을 보여줄 때는 반드시 제목 옆에 '듄: 파트 2 (ID: 693134)' 형식으로 영화 ID를 함께 표기한다."
        "추측으로 답하지 말고 도구 결과에 근거해 답한다."
    )
}]

In [10]:
import httpx

BASE_URL = "https://nomad-movies-2.nomadcoders.workers.dev"

def slim_movie(movie):
    return {key: movie.get(key) for key in ("id", "title", "release_date", "vote_average", "overview")}

def get_popular_movies():
    movies = httpx.get(f"{BASE_URL}/movies", timeout=10).json()
    return [slim_movie(movie) for movie in movies]

def get_movie_details(id):
    return httpx.get(f"{BASE_URL}/movies/{id}", timeout=10).json()

def get_similar_movies(id):
    movies = httpx.get(f"{BASE_URL}/movies/{id}/similar", timeout=10).json()
    return [slim_movie(movie) for movie in movies]

def get_movie_credits(id):
    return httpx.get(f"{BASE_URL}/movies/{id}/credits", timeout=10).json()

FUNCTION_MAP = {
    "get_popular_movies": get_popular_movies,
    "get_movie_details": get_movie_details,
    "get_similar_movies": get_similar_movies,
    "get_movie_credits": get_movie_credits,
}

In [11]:
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "get_popular_movies",
            "description": "현재 인기 영화 목록을 가져온다. 인자 없음",
            "parameters": {
                "type": "object", "properties": {}
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_movie_details",
            "description": "특정 영화 ID를 받아서 영화 상세 정보를 가져온다.",
            "parameters": {
                "type": "object",
                "properties": {
                    "id": {"type": "integer", "description": "영화 ID, 예: 550"}
                },
                "required": ["id"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_similar_movies",
            "description": "특정 영화 ID를 받아서 그 영화와 비슷한 영화 목록을 가져온다. 추천 요청 시 사용한다.",
            "parameters": {
                "type": "object",
                "properties": {
                    "id": {"type": "integer", "description": "영화 ID, 예: 550"}
                },
                "required": ["id"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_movie_credits",
            "description": "특정 영화 ID를 받아서 영화 출연진 및 제작진 정보를 가져온다.",
            "parameters": {
                "type": "object",
                "properties": {
                    "id": {"type": "integer", "description": "영화 ID, 예: 550"}
                },
                "required": ["id"]
            }
        }
    }
]

In [12]:
def execute_tool_call(tool_call):
    function_name = tool_call.function.name

    try:
        arguments = json.loads(tool_call.function.arguments)
    except json.JSONDecodeError:
        arguments = {}

    args_str = ", ".join(str(v) for v in arguments.values())
    print(f"Agent: [{function_name}({args_str}) 호출]")

    function_to_run = FUNCTION_MAP.get(function_name)
    if function_to_run is None:
        return {"error": f"unknown function: {function_name}"}

    try:
        return function_to_run(**arguments)
    except Exception as e:
        return {"error": str(e)}


def call_ai():
    for _ in range(10):
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=messages,
            tools=TOOLS,
        )
        message = response.choices[0].message

        if not message.tool_calls:
            messages.append({"role": "assistant", "content": message.content})
            print(f"Agent: {message.content}")
            return

        messages.append(message)
        for tool_call in message.tool_calls:
            result = execute_tool_call(tool_call)
            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": json.dumps(result, ensure_ascii=False),
            })


In [13]:
while True:
    message = input("Send a message to the LLM...")
    if message == "quit" or message == "q":
        break
    else:
        messages.append({"role": "user", "content": message})
        print(f"User: {message}")
        call_ai()

User: 지금 인기 있는 영화 알려줘
Agent: [get_popular_movies() 호출]
Agent: 현재 인기 있는 영화 목록입니다:

1. **Peddi** (ID: 1057265) - 2026-06-03 출시. 6.466점. 1980년대 인도 아나드라 프라데시에서, 한 마을의 활기찬 주민이 스포츠를 통해 공동체를 단결시킵니다.
   
2. **Obsession** (ID: 1339713) - 2026-05-13 출시. 7.9점. "원 원스 위로"를 부숴서 사랑에 빠진 남자가 희망했던 대로 모든 것을 얻게 되지만, 그 욕망이 어두운 대가를 치르게 됩니다.
   
3. **Hai Jawani Toh Ishq Hona Hai** (ID: 1308553) - 2026-06-04 출시. 5.357점. 결혼을 말아먹은 Jass가 서로를 불행하게 만드는 충격적인 사실과 마주하게 되는 이야기입니다.
   
4. **The Unknown Man** (ID: 879945) - 2021-10-16 출시. 8.2점. 플라망 작가 루이, 창작의 영감을 얻기 위해 코트 다쥐르에 고립된 삶을 선택합니다.
   
5. **Mortal Kombat II** (ID: 931285) - 2026-05-06 출시. 7.947점. 저장된 전투에서 최강의 영웅들이 최종 결투를 벌입니다.
   
6. **Michael** (ID: 936075) - 2026-04-22 출시. 8.506점. 마이클 잭슨의 생애를 조명하며 그의 음악을 넘어선 삶을 탐구합니다.
   
7. **The Mandalorian and Grogu** (ID: 1228710) - 2026-05-20 출시. 6.794점. 새로운 공화국이 사라진 제국을 물리치기 위해 전설의 현상금 사냥꾼과 그의 제자를 활용합니다.
   
8. **The Super Mario Galaxy Movie** (ID: 1226863) - 2026-04-01 출시. 8.167점. 마리오와 루이지가 새로운 위협에 맞서기 위해 별들을 가로지를 준비를 